### 1.) Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_data_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

### 2.) Deduplicação, Limpeza e Tratamento das tabelas

### 2.1) 
### origem: bronze_movies_info
### destino: silver.tb_info_filmes

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

def mostrar_padrao_data(df, datetime_column):
    # Substitui qualquer dígito numérico (0-9) pelo número 9
    # Assim, "25/10/2023" vira "99/99/9999" e "2023-05-12" vira "9999-99-99"
    df_padroes = df.withColumn(
        "padrao_data",
        F.regexp_replace(F.col(datetime_column), r"\d", "9")
    )

    # Agrupa e conta a frequência de cada padrão encontrado no seu dataset
    df_padroes.groupBy("padrao_data").count().orderBy(F.desc("count")).show()

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Verificando padrões de datas:
mostrar_padrao_data(df_bronze_info, "release_date")
# Os formatos identificados foram: MM/dd/yyyy, MM-dd-yyyy e yyyy-MM-dd


df_silver_info = (
    df_bronze_info
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("status_filme", F.upper(F.trim(F.replace(F.col("status_filme"), F.lit("-"), F.lit(" ")))))
    .withColumn("status_filme",
        F.when(F.col("status_filme") == "RELEASED", "Lançado")
        .when(F.col("status_filme") == "POST PRODUCTION", "Pós-Produção")
        .when(F.col("status_filme") == "IN PRODUCTION", " Em Produção")
        .when(F.col("status_filme") == "PLANNED", "Planejado")
        .when(F.col("status_filme") == "RUMORED", "Rumores")
        .when(F.col("status_filme") == "CANCELED", "Cancelado")
        .otherwise("Não Informado"))
    # para retirar duplicatas selecionando a entrada com ingestão mais recente,
    # ordenamos de forma decrescente em relação a ingestion_datetime, pois o spark
    # vai manter a primeira linha que encontrar
    .sort(F.col("ingestion_datetime").desc())
    .drop_duplicates(["id_filme"])
    .withColumn("data_lancamento",
        F.coalesce(
        F.try_to_timestamp("data_lancamento", F.lit("yyyy-MM-dd")),
        F.try_to_timestamp("data_lancamento", F.lit("MM/dd/yyyy")),
        F.try_to_timestamp("data_lancamento", F.lit("MM-dd-yyyy"))
    ))
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))

)

# verificar padrões de data após formatação
mostrar_padrao_data(df_silver_info, "data_lancamento")

# Armazenamento na camada silver
df_silver_info.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_silver_info.limit(10))

+--------------------+------+
|         padrao_data| count|
+--------------------+------+
|          9999-99-99|183082|
|          99/99/9999| 21204|
|          99-99-9999|  9434|
|                NULL|   124|
| his mother then ...|     2|
| northern Sweden....|     2|
| a boy who delive...|     2|
| in partnership w...|     2|
| at least for a b...|     2|
| Manny Pacquiao o...|     2|
| the more Masuda ...|     2|
|World's greatest ...|     2|
+--------------------+------+

+-------------------+-----+
|        padrao_data|count|
+-------------------+-----+
|9999-99-99 99:99:99|92185|
|               NULL| 5694|
+-------------------+-----+



id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ingestion_datetime,ano_lancamento
354912,tt2380307,Coco,Coco,en,2017-10-27T00:00:00.000Z,105,Lançado,"Despite his family’s baffling generations-old ban on music, Miguel dreams of becoming an accomplished musician like his idol, Ernesto de la Cruz. Desperate to prove his talent, Miguel finds himself in the stunning and colorful Land of the Dead following a mysterious chain of events. Along the way, he meets charming trickster Hector, and together, they set off on an extraordinary journey to unlock the real story behind Miguel's family history.",The celebration of a lifetime,2026-10-07T18:20:58.781Z,2017
766507,tt11866324,Prey,Prey,en,2022-08-02T00:00:00.000Z,100,Lançado,"When danger threatens her camp, the fierce and highly skilled Comanche warrior Naru sets out to protect her people. But the prey she stalks turns out to be a highly evolved alien predator with a technically advanced arsenal.",null,2026-10-07T18:20:58.781Z,2022
389015,tt5580036,"I, Tonya","I, Tonya",en,2017-12-08T00:00:00.000Z,120,Lançado,"Competitive ice skater Tonya Harding rises amongst the ranks at the U.S. Figure Skating Championships, but her future in the sport is thrown into doubt when her ex-husband intervenes.",Fitting in is overrated,2026-10-07T18:20:58.781Z,2017
512195,tt7991608,Red Notice,Red Notice,en,2021-11-04T00:00:00.000Z,118,Lançado,"An Interpol-issued Red Notice is a global alert to hunt and capture the world's most wanted. But when a daring heist brings together the FBI's top profiler and two rival criminals, there's no telling what will happen.",Pro and cons.,2026-10-07T18:20:58.781Z,2021
311324,tt2034800,The Great Wall,The Great Wall,en,2016-12-16T00:00:00.000Z,103,Lançado,European mercenaries searching for black powder become embroiled in the defense of the Great Wall of China against a horde of monstrous creatures.,1700 years to build. 5500 miles long. What were they trying to keep out?,2026-10-07T18:20:58.781Z,2016
661374,tt11564570,Glass Onion: A Knives Out Mystery,Glass Onion: A Knives Out Mystery,en,2022-12-11T00:00:00.000Z,140,Lançado,World-famous detective Benoit Blanc heads to Greece to peel back the layers of a mystery surrounding a tech billionaire and his eclectic crew of friends.,"When the game ends, the mystery begins.",2026-10-07T18:20:58.781Z,2022
308266,tt2005151,War Dogs,War Dogs,en,2016-08-18T00:00:00.000Z,114,Lançado,"Based on the true story of two young men, David Packouz and Efraim Diveroli, who won a $300 million contract from the Pentagon to arm America's allies in Afghanistan.",An American dream,2026-10-07T18:20:58.781Z,2016
399035,tt1590193,The Commuter,The Commuter,en,2018-01-11T00:00:00.000Z,104,Lançado,"A businessman, on his daily commute home, gets unwittingly caught up in a criminal conspiracy that threatens not only his life but the lives of those around him.",Lives are on the line,2026-10-07T18:20:58.781Z,2018
438695,tt6467266,SING 2,Sing 2,en,2021-01-12T00:00:00.000Z,110,Lançado,"Buster and his new cast now have their sights set on debuting a new show at the Crystal Tower Theater in glamorous Redshore City. But with no connections, he and his singers must sneak into the Crystal Entertainment offices, run by the ruthless wolf mogul Jimmy Crystal, where the gang pitches the ridiculous idea of casting the lion rock legend Clay Calloway in their show. Buster must embark on a quest to find the now-isolated Clay and persuade him to return to the stage.",Where will your dreams take you?,2026-10-07T18:20:58.781Z,2021
445651,tt4073790,The Darkest Minds,The Darkest Minds,en,2018-07-25T00:00:00.000Z,105,Lançado,"After a disease kills 98% of America's children, the surviving 2% develop superpowers and are placed in internment camps. A 16-year-old girl escapes her camp and joins a group of other teens on the run from the government.",The ones who changed will change everything,2026-10-07T18:20:58

### 2.2)
### origem: bronze.tb_movies_financials  
### destino: silver.tb_financeiro_filmes

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

DEC = DecimalType(18, 2)

# Textos que representam ausência de dado (comparados em minúsculas)
ENTRADAS_NULAS = ["unknown", "não informado", "nao informado", "n/a", "null", "none", ""]


def limpar_valor_monetario(coluna):
    """Converte textos como '$1,500,000', 'R$ 1.500.000,00' ou 'Unknown' em decimal (ou NULL)."""
    txt = F.trim(F.col(coluna).cast("string"))

    # 1. Mantém só dígitos, separadores e sinal (remove $, USD, R$, espaços etc.)
    so_numeros = F.regexp_replace(txt, r"[^0-9,.\-]", "")

    tem_virgula = so_numeros.contains(",")
    tem_ponto = so_numeros.contains(".")

    # 2. Normaliza separadores para o padrão 1234567.89
    normalizado = (
        F.when(tem_virgula & tem_ponto & so_numeros.rlike(r",\d+$"),         # 1.500.000,00 (BR)
               F.regexp_replace(F.regexp_replace(so_numeros, r"\.", ""), ",", "."))
         .when(tem_virgula & tem_ponto,                                         # 1,500,000.00 (US)
               F.regexp_replace(so_numeros, ",", ""))
         .when(tem_virgula & so_numeros.rlike(r"^-?\d{1,3}(,\d{3})+$"),         # 1,500,000
               F.regexp_replace(so_numeros, ",", ""))
         .when(tem_virgula,                                                     # 1500,50
               F.regexp_replace(so_numeros, ",", "."))
         .when(tem_ponto & so_numeros.rlike(r"^-?\d{1,3}(\.\d{3})+$"),          # 1.500.000
               F.regexp_replace(so_numeros, r"\.", ""))
         .otherwise(so_numeros)                                                 # 1500000 ou 1500.50
    )

    # 3. Textos de ausência -> NULL antes da conversão; try_cast devolve NULL para lixo residual
    valor = (
        F.when(txt.isNull() | F.lower(txt).isin(ENTRADAS_NULAS), F.lit(None).cast(DEC))
         .otherwise(normalizado.try_cast(DEC))
    )

    # 4. Zero ou negativo -> ausente
    return F.when(valor > 0, valor).otherwise(F.lit(None).cast(DEC))


# Cotação mais recente disponível na bronze
ultima_cotacao = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.desc("dataHoraCotacao"))
    .select("cotacaoCompra", "dataHoraCotacao")
    .first()
)
if ultima_cotacao is None:
    raise ValueError("tb_cotacao_dolar está vazia: rode a ingestão da API antes da silver.")

taxa = F.lit(ultima_cotacao["cotacaoCompra"]).cast(DecimalType(10, 4))
print(f"Cotação aplicada: {ultima_cotacao['cotacaoCompra']} ({ultima_cotacao['dataHoraCotacao']})")


# Transformação silver
df_bronze_financials = spark.table(f"{bronze_schema}.tb_movies_financials")

df_silver_financials = (
    df_bronze_financials
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    # Higienização + conversão para decimal + zero/negativo -> NULL
    .withColumn("orcamento_usd", limpar_valor_monetario("orcamento_usd"))
    .withColumn("receita_usd", limpar_valor_monetario("receita_usd"))

    # Conversão para real
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * taxa, 2).cast(DEC))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * taxa, 2).cast(DEC))

    # Lucro só é calculado se houver receita e orçamento não nulos
    .withColumn(
        "lucro_usd",
        F.when(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd")).cast(DEC)
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("receita_brl").isNotNull() & F.col("orcamento_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl")).cast(DEC)
    )

    # Margem % = lucro / receita * 100, evitando divisão por zero/NULL
    .withColumn(
        "margem_lucro_pct",
        F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
               F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast(DEC)
    )

    .select(
        "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl", "margem_lucro_pct"
    )
)

display(df_silver_financials)

# Armazenamento na camada silver
df_silver_financials.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

Cotação aplicada: 4.9929 (2026-10-07 13:05:23.249063)


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
293660,58000000.00,null,289588200.00,null,null,null,null
299536,300000000.00,2052415039.00,1497870000.00,10247503048.22,1752415039.00,8749633048.22,85.38
299534,356000000.00,2800000000.00,1777472400.00,13980120000.00,2444000000.00,12202647600.00,87.29
475557,55000000.00,1074458282.00,274609500.00,5364662756.20,1019458282.00,5090053256.20,94.88
271110,250000000.00,null,1248225000.00,null,null,null,null
284054,200000000.00,1349926083.00,998580000.00,6740045939.81,1149926083.00,5741465939.81,85.18
284052,180000000.00,676343174.00,898722000.00,3376913833.46,496343174.00,2478191833.46,73.39
315635,175000000.00,880166924.00,873757500.00,4394585434.84,705166924.00,3520827934.84,80.12
283995,200000000.00,863756051.00,998580000.00,4312647587.04,663756051.00,3314067587.04,76.85
297761,175000000.00,746846894.00,873757500.00,3728931857.05,571846894.00,2855174357.05,76.57


### 2.3)
### origem: bronze.tb_movies_metrics
### destino: silver.tb_metricas_engajamento

In [0]:
df_bronze_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")

display(df_bronze_metrics.limit(10))

df_silver_metrics = (
    df_bronze_metrics
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
    .withColumn("popularidade", F.regexp_replace(F.trim(F.col("popularidade")), ",", ".").try_cast("double"))
    .withColumn("nota_media_tmdb", F.col("nota_media_tmdb").try_cast("double"))
    .withColumn("qtd_votos_tmdb", F.col("qtd_votos_tmdb").try_cast("integer"))
    .withColumn("nota_media_imdb", F.col("nota_media_imdb").try_cast("double"))
    .withColumn("qtd_votos_imdb", F.col("qtd_votos_imdb").try_cast("integer"))
    .withColumn("popularidade",
                F.when(F.col("popularidade")<0, None).otherwise(F.col("popularidade"))
    )
    .withColumn("qtd_votos_tmdb",
                F.when(F.col("qtd_votos_tmdb")<0, None).otherwise(F.col("qtd_votos_tmdb"))
    )
    .withColumn("qtd_votos_imdb",
                F.when(F.col("qtd_votos_imdb")<0, None).otherwise(F.col("qtd_votos_imdb"))
    )
    .withColumn("nota_media_tmdb",
                F.when(F.col("nota_media_tmdb")<0, None).when(F.col("nota_media_tmdb")>10, None).otherwise(F.col("nota_media_tmdb"))
    )
    .withColumn("nota_media_imdb",
                F.when(F.col("nota_media_imdb")<0, None).when(F.col("nota_media_imdb")>10, None).otherwise(F.col("nota_media_imdb"))
    )
)

# Armazenamento na camada silver
df_silver_metrics.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_metrics.filter(F.col("popularidade").isNull()).limit(10))

id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T14:15:34.966Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T14:15:34.966Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T14:15:34.966Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T14:15:34.966Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T14:15:34.966Z
284054,43.665,7.39,null,7.3,924922,2026-10-07T14:15:34.966Z
284052,70.535,7.427,20935,7.5,895880,2026-10-07T14:15:34.966Z
315635,65.88,7.345,20507,7.4,835116,2026-10-07T14:15:34.966Z
283995,67.553,7.624,20353,7.6,844767,2026-10-07T14:15:34.966Z
297761,35.356,5.909,20097,5.9,null,2026-10-07T14:15:34.966Z


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
324857,null,8.404,null,null,null,2026-10-07T18:21:01.525Z
301528,null,7.5,9093,null,null,2026-10-07T18:21:01.525Z
615457,null,8.018,6217,null,null,2026-10-07T18:21:01.525Z
390043,null,null,null,null,null,2026-10-07T18:21:01.525Z
140300,null,6.878,5327,null,null,2026-10-07T18:21:01.525Z
397243,null,6.755,null,null,null,2026-10-07T18:21:01.525Z
364689,null,7.232,2652,null,null,2026-10-07T18:21:01.525Z
594767,null,6.662,2416,null,null,2026-10-07T18:21:01.525Z
505948,null,6.684,2345,null,null,2026-10-07T18:21:01.525Z
602269,null,6.329,2158,null,null,2026-10-07T18:21:01.525Z


### 2.4)
### origem: bronze.tb_movies_reviews  
### destino: silver.tb_avaliacoes_usuarios

In [0]:
df_bronze_reviews = spark.table(f"{bronze_schema}.tb_movies_reviews")

display(df_bronze_reviews.limit(5))

df_silver_reviews = (
    df_bronze_reviews
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
    .withColumn("nota_usuario",
        F.when(F.col("nota_usuario")<0, None).when(F.col("nota_usuario")>10, None).otherwise(F.col("nota_usuario"))
    )
    .withColumn("comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == ""), "Sem comentário"
    ).otherwise(F.col("comentario_usuario")))
)

# Armazenamento na camada silver
df_silver_reviews.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_reviews.limit(5))

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-07T14:15:37.323Z
637007,Lucas Reis 602,3.9,null,2026-10-07T14:15:37.323Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-07T14:15:37.323Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-07T14:15:37.323Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-07T14:15:37.323Z


id_filme,nome_usuario,nota_usuario,comentario_usuario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,Sem comentário,2026-10-07T18:21:03.542Z
1152100,Isabela Pinto 509,7.6,Legal! Uma boa opção para o fim de semana.,2026-10-07T18:21:03.542Z
846452,Sérgio Freitas,6.1,Assisti até o final mas não me marcou.,2026-10-07T18:21:03.542Z
410937,Carolina Lopes 276,2.1,Horrível! Perda de tempo.,2026-10-07T18:21:03.542Z
760653,Maria Barbosa 699,3.5,Sem comentário,2026-10-07T18:21:03.542Z


### 2.5)
### origem: bronze.tb_credits_and_tags, coluna genres
### destino: silver.tb_generos

In [0]:
df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

display(df_bronze_credits.limit(5))

GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

df_silver_generos = (
    df_bronze_credits
    .select("genres")

    # 1. Padroniza o separador: troca ";" e "|" por ","
    .withColumn("genres", F.regexp_replace("genres", "[;|]", ","))

    # 2. Split + explode: uma linha por gênero
    .withColumn("genero", F.explode(F.split("genres", ",")))

    # 3. Mantém apenas valores do domínio de gêneros
    .filter(F.col("genero").isin(GENEROS_VALIDOS))

    # 4. Remove resíduos
    .filter(F.col("genero") != "")                       # vazios (separadores extras)
    .filter(~F.col("genero").rlike(r"^[0-9.,\-]+$"))     # números deslocados
    .withColumn("genero", F.trim(F.regexp_replace("genero", '"', "")))  # retira aspas das strings
    .filter(F.length("genero") <= 20)                    # elimina textos muito longos
    .select("genero")
    .dropDuplicates()
)

display(df_silver_generos)

# Armazenamento na camada silver
df_silver_generos.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_generos")

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T14:15:26.171Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T14:15:26.171Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T14:15:26.171Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T14:15:26.171Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T14:15:26.171Z


genero
Fantasy
Thriller
Adventure
Drama
Western
Crime
Horror
Documentary
Mystery
War


2.6)  
origem: bronze.tb_credits_and_tags (cast, directors, writers, e production_companies)  
destino: silver.tb_pessoas_empresas
 
Nessa tabela, a ideia é unir nomes de pessoas e de companhias, adicionando a coluna tipo_entidade para identificar esse nome.

In [0]:
df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")


def explodir_entidade(coluna, tipo):
    """Gera uma linha por pessoa/empresa da coluna, com o tipo de entidade."""
    return (
        df_bronze_credits
        .select(F.col(coluna).alias("nome"))
        .withColumn("nome", F.regexp_replace("nome", "[;|]", ","))
        .withColumn("nome", F.explode(F.split("nome", ",")))
        .withColumn("nome", F.initcap(F.trim("nome"))) # apenas a primeira letra de cada palavra maiúscula
        .withColumn("tipo_entidade", F.lit(tipo))
    )


df_silver_pessoas_empresas = (
    explodir_entidade("cast", "Ator")
    .unionByName(explodir_entidade("directors", "Diretor"))
    .unionByName(explodir_entidade("writers", "Roteirista"))
    .unionByName(explodir_entidade("production_companies", "Produtora"))

    # Remove resíduos
        # Remove resíduos
    .filter(F.col("nome") != "")                                # vazios
    .filter(~F.col("nome").rlike(r"^[0-9.,\-]+$"))              # números deslocados
    .filter(~F.col("nome").startswith("/"))                     # caminhos de imagem (.jpg)
    .filter(F.size(F.split("nome", " ")) <= 6)                  # trechos de sinopse

    .dropDuplicates(["nome", "tipo_entidade"])
)

display(df_silver_pessoas_empresas)

# Armazenamento na camada silver
df_silver_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

nome,tipo_entidade
Lupita Nyong'o,Ator
Sophia Lillis,Ator
Kevin Spacey,Ator
Lily James,Ator
Ted Levine,Ator
Lizzy Caplan,Ator
John Malkovich,Ator
Kyle Chandler,Ator
Luke Kirby,Ator
Eric Stonestreet,Ator


### 2.7)
### origem: bronze.tb_cotacao_dolar
### destino: silver.tb_cotacao_dolar

#### Para fazer o preenchimento das datas sem cotação (sábados, domingos e feriados), vamos realizar 4 passos:
#### 1. pegamos as cotações por data da tabela da camada bronze e verificamos a data mínima ('inicio') e a data máxima armazenadas ('fim')
#### 2. geramos uma sequência de datas entre a data inicio e a data fim com todas as datas, inclusive sábados, domingos e feriados, e explodimos essa sequência em um novo dataframe (df_calendario)
#### 3. fazemos join com o dataframe da camada bronze (que contem cotação para os dias úteis)
#### 4. utilizando Window, selecionamos a última cotação disponível

In [0]:
from pyspark.sql.window import Window

# 1. Cotações da bronze, uma por dia
df_cotacao = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .select(
        F.to_date("dataHoraCotacao").alias("data"),
        F.col("cotacaoCompra").alias("cotacao_compra"),
    )
    .dropDuplicates(["data"])
)

# 2. Calendário com todos os dias entre a primeira e a última cotação
df_calendario = (
    df_cotacao
    .select(F.min("data").alias("inicio"), F.max("data").alias("fim"))
    .select(F.explode(F.sequence("inicio", "fim")).alias("data"))
)

# 3. Junta o calendário com as cotações (dias sem cotação ficam NULL)
df_serie = df_calendario.join(df_cotacao, on="data", how="left")

# 4. Forward fill: repete a última cotação não nula
janela = Window.orderBy("data").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = df_serie.withColumn(
    "cotacao_compra",
    F.last("cotacao_compra", ignorenulls=True).over(janela),
)

display(df_silver_cotacao.orderBy("data"))

# Armazena na camada silver
df_silver_cotacao.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data,cotacao_compra
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692
2026-10-07,4.9929
